# 01 Baseline Ml

Thin notebook over the phase script(s) in `scripts/`. Cells are split at the `# ----` section markers. Run from the repo root (`jupyter lab`), or just run the scripts.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline

## scripts/10_phase1.py

In [ ]:
"""Phase 1: two independent estimates of promotional incremental sales, with bootstrap CIs.

Outputs
  data/processed/predictions.parquet   counterfactual units & hhs for every store-product-week
  data/processed/events.parquet        one row per promotion run with Method A and Method B estimates
  outputs/phase1_results.md            validation numbers, comparison tables
  outputs/figures/phase1_*.png         parallel trends, placebo, A vs B
"""
import json
import sys
from pathlib import Path

import matplotlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getcwd())
sys.path.insert(0, str(ROOT))
from src.load import build_clean, PROC  # noqa: E402
from src import baseline as B  # noqa: E402
from src.panel import build_panel, events_table  # noqa: E402
from src.did import run_did, parallel_trends, MIN_CONTROLS  # noqa: E402
from src.bootstrap import summarise, boot_ci  # noqa: E402

FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
OUT = ROOT / "outputs" / "phase1_results.md"
N_BOOT = 1000
lines = []; P = lines.append

df = build_clean()

In [ ]:
# ------------------------------------------------------------------ Method A
P("# Phase 1 results (generated by scripts/10_phase1.py)\n")
P("## Method A: gradient-boosting counterfactual trained on clean non-promo weeks\n")
train_mask = B.training_mask(df, clean_gap=4)
nonpromo = (~df.is_promo).sum()
P(f"- Non-promo rows: {nonpromo:,}. Of these, **{train_mask.sum():,} ({train_mask.sum()/nonpromo:.0%})** are 'clean' "
  "(more than 4 weeks after the last promotion) and form the training set. Post-promo weeks are excluded "
  "because they carry the pantry-loading dip Phase 2 measures.")
feat = B.add_features(df, train_mask)
models, vals = {}, {}
for target in ["units", "hhs"]:
    models[target], vals[target] = B.fit_and_validate(feat, train_mask, target)
preds = B.predict_all(feat, models)
preds.to_parquet(PROC / "predictions.parquet", index=False)

P("\n### Validation on a held-out 20% of clean non-promo weeks\n")
vt = pd.DataFrame([v.as_dict() for v in vals.values()])
vt = vt.rename(columns={"rmse_naive": "rmse_naive_woy", "mape_naive": "mape_naive_woy"})
P(vt.to_markdown(index=False, floatfmt=".3f"))
u = vals["units"]
P(f"\n- Units: model RMSE {u.rmse_model:.2f} vs naive store-product-week-of-year median {u.rmse_naive:.2f} "
  f"({1-u.rmse_model/u.rmse_naive:+.0%}); MAPE {u.mape_model:.1%} vs {u.mape_naive:.1%}. "
  f"The model {'beats' if u.rmse_model < u.rmse_naive else 'does NOT beat'} the naive benchmark.")
P("- A second naive benchmark (store-product median, no seasonality) is shown as `*_naive_sp`.")

# error on post-promo weeks (not in training): does the model see the dip?
post = df[(~df.is_promo) & (df.weeks_since_promo <= 4)]
pp = post[["store_num", "upc", "week_index", "units"]].merge(preds, on=["store_num", "upc", "week_index"])
P(f"\n- Aggregate calibration on the clean holdout: actual/predicted = **{u.calibration:.3f}** (1.0 = unbiased).")
P(f"- Sanity check on **post-promo weeks (1-4 weeks after a promotion, never trained on)**: actual/predicted = "
  f"**{pp.units.sum()/pp.pred_units.sum():.3f}**. Relative to the holdout calibration this is "
  f"**{pp.units.sum()/pp.pred_units.sum()/u.calibration:.3f}**. Below 1 means these weeks sell less than an "
  "undisturbed week (a pantry-loading dip); above 1 means, in aggregate, they do not. Phase 2 looks at this per event and per category.")

In [ ]:
# ------------------------------------------------------------------ events + Method A per event
ev = events_table(df)
pan = build_panel(df, preds)
pa = df[df.is_promo].merge(preds, on=["store_num", "upc", "week_index"])
agg = pa.groupby("promo_run_id").agg(baseline_units=("pred_units", "sum"), baseline_hhs=("pred_hhs", "sum")).reset_index()
agg["event_id"] = agg["promo_run_id"].astype("int64")
ev = ev.merge(agg.drop(columns="promo_run_id"), on="event_id", how="left")
ev["incr_A"] = ev["actual_units"] - ev["baseline_units"]
ev["lift_A_pct"] = ev["incr_A"] / ev["baseline_units"]
ev["incr_A_per_week"] = ev["incr_A"] / ev["run_length"]

In [ ]:
# ------------------------------------------------------------------ Method B
did = run_did(pan, ev)
ev = ev.merge(did, on="event_id", how="left")
ev["incr_B"] = ev["did_incr"]
ev["lift_B_pct"] = ev["did_incr"] / ev["did_counterfactual"]
ev.to_parquet(PROC / "events.parquet", index=False)

P("\n## Method B: difference-in-differences against same-product control stores\n")
P(f"- {len(ev):,} promotion runs (events). {ev.pre_n.ge(4).mean():.0%} have at least 4 clean pre-weeks; "
  f"**{ev.has_controls.mean():.0%} ({ev.has_controls.sum():,}) have at least {MIN_CONTROLS} valid control stores** "
  "(same product, not promoted during the run, at least 4 clean pre-weeks).")
P(f"- Median number of controls when available: {ev.loc[ev.has_controls,'n_controls'].median():.0f}.")
P("- Events without controls are mostly chain-wide feature/price events; this is a finding about the promotion "
  "mechanism, not a failure of the method. Method A covers them.")

# parallel trends
pt = parallel_trends(pan, ev, did)
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(pt.rel_week, pt.treated, yerr=1.96 * pt.treated_se, marker="o", label="treated store", color="#c0392b")
ax.errorbar(pt.rel_week, pt.control, yerr=1.96 * pt.control_se, marker="s", label="control stores (mean)", color="#1f6f8b")
ax.axhline(1, ls="--", color="grey", lw=.8)
ax.set_xlabel("weeks before promotion start"); ax.set_ylabel("units / own clean pre-window mean")
ax.set_title(f"Parallel trends check, {int(pt.n_events.max()):,} events with controls"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase1_parallel_trends.png", dpi=130); plt.close(fig)
gap = (pt.treated - pt.control).abs().max()
P(f"\n### Parallel trends\n\n- Max absolute gap between treated and control indexed series over t-8..t-1: **{gap:.3f}** "
  f"(both are indexed to 1.0 by construction on average). Figure: outputs/figures/phase1_parallel_trends.png")
P(pt.to_markdown(index=False, floatfmt=".3f"))

# placebo
pl = ev.dropna(subset=["placebo_did"])
m, lo, hi, n = boot_ci(pl.placebo_did.to_numpy(), n_boot=N_BOOT)
mr = pl.placebo_did.sum() / pl.placebo_cf.sum()
real_m = ev.loc[ev.has_controls, "incr_B_per_week"] if "incr_B_per_week" in ev else ev.loc[ev.has_controls, "incr_B"] / ev.loc[ev.has_controls, "run_length"]
P(f"\n### Placebo test at t-3\n\n- {n:,} placebo events. Mean placebo DiD = **{m:.2f} units/week** (95% CI {lo:.2f} to {hi:.2f}); "
  f"as a share of the counterfactual: **{mr:+.1%}**. Compare with the real per-week DiD effect of {real_m.mean():.2f} units/week. "
  f"The placebo is {abs(m)/real_m.mean():.0%} of the real effect.")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(pl.placebo_did.clip(-50, 50), bins=60, color="#1f6f8b", alpha=.8)
ax.axvline(0, color="k", lw=1); ax.axvline(m, color="#c0392b", ls="--", label=f"mean {m:.2f}")
ax.set_xlabel("placebo DiD at t-3 (units, clipped to +/-50)"); ax.set_ylabel("events"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase1_placebo.png", dpi=130); plt.close(fig)

In [ ]:
# ------------------------------------------------------------------ compare A and B
P("\n## Method A vs Method B\n")
both = ev[ev.has_controls].copy()
sa = summarise(ev, "incr_A", "baseline_units", n_boot=N_BOOT); sa["method"] = "A (all events)"
sa2 = summarise(both, "incr_A", "baseline_units", n_boot=N_BOOT); sa2["method"] = "A (events with controls)"
sb = summarise(both, "incr_B", "did_counterfactual", n_boot=N_BOOT); sb["method"] = "B (events with controls)"
overall = pd.concat([sa, sa2, sb])[["method", "n_events", "mean_incr_units", "incr_lo", "incr_hi", "lift_pct", "lift_lo", "lift_hi"]]
P("### Overall (per-event totals over the run; lift % = sum incremental / sum baseline)\n")
P(overall.to_markdown(index=False, floatfmt=".3f"))

ca = summarise(both, "incr_A", "baseline_units", by=["category"], n_boot=N_BOOT); ca["method"] = "A"
cb = summarise(both, "incr_B", "did_counterfactual", by=["category"], n_boot=N_BOOT); cb["method"] = "B"
cat = pd.concat([ca, cb]).sort_values(["category", "method"])[["category", "method", "n_events", "mean_incr_units", "incr_lo", "incr_hi", "lift_pct", "lift_lo", "lift_hi"]]
P("\n### By category, same events (those with valid controls)\n")
P(cat.to_markdown(index=False, floatfmt=".3f"))

pa_ = summarise(both, "incr_A", "baseline_units", by=["upc", "description"], n_boot=300); pa_["method"] = "A"
pb_ = summarise(both, "incr_B", "did_counterfactual", by=["upc", "description"], n_boot=300); pb_["method"] = "B"
prod = pa_.merge(pb_, on=["upc", "description"], suffixes=("_A", "_B"))
prod = prod[["upc", "description", "n_events_A", "lift_pct_A", "lift_lo_A", "lift_hi_A", "lift_pct_B", "lift_lo_B", "lift_hi_B"]].sort_values("lift_pct_A", ascending=False)
prod["upc"] = prod["upc"].astype(str)
P("\n### By product, same events\n")
P(prod.to_markdown(index=False, floatfmt=".3f"))
prod.to_csv(ROOT / "outputs" / "phase1_lift_by_product.csv", index=False)
cat.to_csv(ROOT / "outputs" / "phase1_lift_by_category.csv", index=False)
overall.to_csv(ROOT / "outputs" / "phase1_overall.csv", index=False)

# scatter A vs B per event
fig, ax = plt.subplots(figsize=(5, 5))
lim = np.nanpercentile(np.r_[both.incr_A, both.incr_B], [1, 99])
ax.scatter(both.incr_A, both.incr_B, s=4, alpha=.25, color="#1f6f8b")
ax.plot(lim, lim, color="k", lw=1, ls="--"); ax.set_xlim(lim); ax.set_ylim(lim)
r = np.corrcoef(both.incr_A, both.incr_B)[0, 1]
ax.set_xlabel("Method A incremental units (run total)"); ax.set_ylabel("Method B (DiD) incremental units")
ax.set_title(f"Per-event agreement, r = {r:.2f}"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG / "phase1_A_vs_B.png", dpi=130); plt.close(fig)
P(f"\n- Per-event correlation between A and B: **{r:.2f}** (figure: outputs/figures/phase1_A_vs_B.png).")
diff = sa2.lift_pct.iloc[0] - sb.lift_pct.iloc[0]
P(f"- On the same events, Method A lift is {sa2.lift_pct.iloc[0]:.1%} and Method B is {sb.lift_pct.iloc[0]:.1%} "
  f"(A minus B = {diff:+.1%}). See docs/decision_log.md for which one I trust and why.")

json.dump({"validation": {k: v.as_dict() for k, v in vals.items()},
           "n_events": int(len(ev)), "share_with_controls": float(ev.has_controls.mean()),
           "placebo_mean": m, "placebo_lo": lo, "placebo_hi": hi, "placebo_share": float(mr),
           "post_promo_ratio": float(pp.units.sum() / pp.pred_units.sum()),
           "overall": overall.to_dict(orient="records")},
          open(ROOT / "outputs" / "phase1_summary.json", "w"), indent=2, default=float)
OUT.write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))